# Test 1 (T_n, KS-tipo) — Exploración

Notebook de inspección y depuración del Test 1 (Schuster-Barker).

Carga el módulo `src/`, genera unas muestras de ejemplo, calcula el estadístico T_n, ejecuta el bootstrap y muestra las curvas básicas. Para correr el estudio completo, usar `scripts/run_test1_simulation.py`.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.statistics_ks import Tn_statistic, theta_argmin, theta_median, theta_trimmed
from src.bootstrap_ks import bootstrap_test_Tn
from src.distributions import default_h0_specs, default_ha_specs

## 1. Estadístico T_n(theta) como función de theta

Bajo H0 (muestra simétrica), T_n(theta) debe tener un mínimo cercano al centro verdadero. Bajo H_a, ningún theta logra anular T_n.

In [ ]:
rng = np.random.default_rng(7)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, spec, color in zip(axes, [default_h0_specs()[1], default_ha_specs()[0]], ['C0', 'C3']):
    x = spec.sampler(120, rng)
    th_grid = np.linspace(x.min(), x.max(), 200)
    Tn_grid = [Tn_statistic(x, th) for th in th_grid]
    ax.plot(th_grid, Tn_grid, color=color, lw=2)
    ax.axvline(theta_argmin(x), ls='--', color='black', label=f'argmin')
    ax.axvline(theta_median(x), ls=':', color='gray', label=f'mediana')
    ax.set_title(f'{spec.name} (H0={spec.under_h0})')
    ax.set_xlabel('theta')
    ax.set_ylabel('T_n(theta)')
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('T_n(θ) como función de θ — H0 vs H_a', y=1.02)
plt.tight_layout()

## 2. Distribución bootstrap de T_n bajo H0 y H_a

In [ ]:
rng = np.random.default_rng(11)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, spec, color in zip(axes, [default_h0_specs()[1], default_ha_specs()[0]], ['C0', 'C3']):
    x = spec.sampler(100, rng)
    res = bootstrap_test_Tn(x, estimator='median', B=500, rng=rng)
    ax.hist(res.boot_statistics, bins=30, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(res.statistic, color='black', ls='--', lw=2, label=f'T_n obs = {res.statistic:.2f}')
    ax.set_title(f'{spec.name}  (p={res.p_value:.3f})')
    ax.set_xlabel('T_n*')
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()

## 3. Cargar resultados de la simulación completa

Una vez ejecutado `scripts/run_test1_simulation.py`, los CSV están en `results/data/`.

In [ ]:
import pandas as pd
csv_path = ROOT / 'results' / 'data' / 'tn_simulation_summary.csv'
if csv_path.exists():
    summary = pd.read_csv(csv_path)
    display(summary.pivot_table(index=['dist', 'estimator'], columns='n', values='reject_rate').round(3))
else:
    print('Aún no existe. Correr antes scripts/run_test1_simulation.py')